# Exercise 5 · Write the evaluation report

**Part 06 · Benchmarking and evaluation** · 10 minutes

Decide whether your model should be **deployed**, run in **shadow** mode, or **not deployed**, and back
it with numbers you computed yourself.

Data: `outputs/predictions_demo5.jsonl` from Demo 5. If you have not run Demo 5, the notebook uses the
saved benchmark predictions (Jev, Kev-4B, Laya on AG News) instead, and you evaluate Kev-4B.

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import json
import pandas as pd
from dmw import metrics, checks, bench

DEMO5 = config.OUTPUTS / "predictions_demo5.jsonl"
if DEMO5.exists():
    SOURCE, P = "demo5", pd.read_json(DEMO5, lines=True)
    MODEL, OTHER = "fine-tuned", ("jev" if "jev" in set(pd.read_json(DEMO5, lines=True).model) else "base")
else:
    SOURCE = "benchmark_replay"
    f = bench.frame("ag_news")
    P = pd.concat([pd.DataFrame({"model": m, "id": f.id, "group": "ag_news", "expected": f.gold,
                                 "choice": f[m], "p_top": f[f"{m}_p"], "latency_ms": f[f"{m}_ms"]})
                   for m in ("jev", "kev-4b", "laya")])
    P["review"] = P.p_top.fillna(0) < 0.8
    MODEL, OTHER = "kev-4b", "jev"
print("SOURCE =", SOURCE, "| evaluating", MODEL, "against", OTHER)
P.groupby("model").size()

## 1 · Metrics

In [ ]:
d = P[P.model == MODEL]
accuracy = metrics.accuracy(d.expected.tolist(), d.choice.tolist())
macro_f1 = metrics.macro_f1(d.expected.tolist(), d.choice.tolist())
nf = d[~d.review]
accuracy_not_flagged = metrics.accuracy(nf.expected.tolist(), nf.choice.tolist())
print(f"accuracy {accuracy:.3f} | macro-F1 {macro_f1:.3f} | accuracy when not flagged {accuracy_not_flagged:.3f} "
      f"| review rate {d.review.mean():.0%}")

## 2 · Coverage curve

In [ ]:
curve = metrics.coverage_curve(d.p_top.fillna(0).tolist(), (d.choice == d.expected).tolist())
curve

## 3 · Disagreements

In [ ]:
w = P.pivot(index="id", columns="model", values="choice").join(P[P.model == MODEL].set_index("id")[["expected"]])
dis = w[w[MODEL] != w[OTHER]]
dis

## 4 · Report

In [ ]:
# Pick the lowest threshold that already reaches the fewest automatic errors (keeps the most coverage)
best = curve[curve.automatic_errors == curve.automatic_errors.min()].iloc[0]
examples = dis.head(3).reset_index()
REPORT = {
    "decision": "shadow",
    "threshold": float(best.threshold),
    "accuracy": accuracy, "macro_f1": macro_f1, "accuracy_not_flagged": accuracy_not_flagged,
    "disagreements": [
        {"id": r["id"], "who_is_right": ("ours" if r[MODEL] == r["expected"] else OTHER if r[OTHER] == r["expected"] else "neither"),
         "likely_cause": "label wording overlaps between options; needs a template for this confusion"}
        for _, r in examples.iterrows()
    ],
    "missing_evidence": ("accuracy on a few hundred real, labeled messages from production traffic, the review "
                         "rate at the chosen threshold, and how often unflagged errors happen per week"),
}
print(json.dumps(REPORT, indent=1, default=str))
checks.check_report(REPORT, checks.recompute(P, MODEL))

### Why "shadow"
A 48-case synthetic test set (or 100 AG News examples) can show that the model is promising, not that
it is safe to automate. Shadow mode collects exactly the missing evidence: real inputs, real labels,
and the review workload at the chosen threshold, without acting on any answer.